# 02 Data Cleaning (BUSA 310 Project 1)

**Input:** Philadelphia OPA Properties CSV (raw, unchanged), in `data/raw/`.
**Output:** `data/processed/philly_sf_sales_clean.csv` plus `data/processed/cleaning_log.csv`.

The cleaning rules below are the group's decisions (made 2026-10-08). The code applies them in order and
records how many rows each step removes, so every exclusion can be traced.

Steps 1 and 2 only organize the data (privacy, data types). Steps 3 to 6 apply the group's rules.
Step 7 adds flag columns for issues the group chose to keep, so they can be revisited without re-running the whole pipeline.

AI use note: code drafted with Claude (Claude Code) from the group's stated rules; see AI Use Log.

## 0. Setup and the group's rules

In [ ]:
import glob, os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)


RULES = {
    "category": "SINGLE FAMILY",       
    "min_sale_date": "2024-01-01",     
    "min_sale_price": 25_000,          
    "drop_condo_and_vacant": False,    
    "drop_bundled_sales": False,       
}

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
RAW_DIR, OUT_DIR = "data/raw", "data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

candidates = sorted(glob.glob(f"{RAW_DIR}/opa_properties_public*.csv"))
RAW_PATH = candidates[-1] if candidates else "opa_properties_public.csv"
print("raw file:", RAW_PATH)

raw file: data/raw/opa_properties_public_2026-10-08.csv


## 1. Load raw data and start the cleaning log

In [ ]:
raw = pd.read_csv(RAW_PATH, low_memory=False)
print("raw shape:", raw.shape)

log = []
def record(step, rule, before, after):
    """Add one row to the cleaning log and print it."""
    log.append({"step": step, "rule": rule, "rows_before": before,
                "rows_removed": before - after, "rows_after": after})
    print(f"{step:<34} removed {before - after:>8,}  ->  {after:>8,} rows")

df = raw.copy()          
record("1. Load raw file", "none", len(df), len(df))

raw shape: (583780, 78)


1. Load raw file                   removed        0  ->   583,780 rows


## 2. Organize: privacy and data types
- Drop owner names and mailing address (personal data, not needed for prediction).
- Turn the map geometry column into two coordinate columns (`x_2272`, `y_2272`, PA State Plane feet), used later to place each home in its census tract and school catchment.
- `sale_date` is stored in UTC (04:00 or 05:00 = midnight Philadelphia time), so convert to Eastern time before taking the date.
- IDs and ZIP codes become text, so they cannot be treated as numbers by a model.

In [ ]:
xy = df["shape"].str.extract(r"POINT\(([-\d.]+) ([-\d.]+)\)").astype(float)
df["x_2272"], df["y_2272"] = xy[0], xy[1]

PERSONAL = ["owner_1", "owner_2", "mailing_address_1", "mailing_address_2", "mailing_care_of",
            "mailing_city_state", "mailing_street", "mailing_zip", "shape"]
df = df.drop(columns=[c for c in PERSONAL if c in df.columns])

for col in ["sale_date", "assessment_date", "recording_date"]:
    df[col] = (pd.to_datetime(df[col], errors="coerce", utc=True)
                 .dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize())

df["parcel_number"] = df["parcel_number"].astype("Int64").astype(str)
df["zip_code"] = df["zip_code"].astype("Int64").astype(str).str.zfill(5).replace("<NA>", np.nan)
df["census_tract"] = df["census_tract"].astype("Int64").astype(str).replace("<NA>", np.nan)

text_cols = df.select_dtypes(include=["object", "string"]).columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())

print("columns after dropping personal data:", df.shape[1], "| homes without coordinates:", int(df["x_2272"].isna().sum()))
print("sale_date dtype:", df["sale_date"].dtype, "| unparseable sale dates:", df["sale_date"].isna().sum())
record("2. Organize (no rows removed)", "drop personal columns, fix types", len(df), len(df))

columns after dropping personal data: 71 | homes without coordinates: 103
sale_date dtype: datetime64[us] | unparseable sale dates: 2334
2. Organize (no rows removed)      removed        0  ->   583,780 rows


## 3. Keep the OPA category the group chose

In [4]:
b = len(df)
df = df[df["category_code_description"] == RULES["category"]]
record("3. Category filter", f"category_code_description == {RULES['category']}", b, len(df))

3. Category filter                 removed  120,771  ->   463,009 rows


## 4. Keep rows with a valid sale date

In [5]:
b = len(df)
df = df[df["sale_date"].notna()]
record("4. Valid sale date", "sale_date not missing or unparseable", b, len(df))

4. Valid sale date                 removed    1,202  ->   461,807 rows


## 5. Sale date window

In [6]:
b = len(df)
df = df[df["sale_date"] >= pd.Timestamp(RULES["min_sale_date"])]
record("5. Sale date window", f"sale_date >= {RULES['min_sale_date']}", b, len(df))
print("date range kept:", df["sale_date"].min().date(), "to", df["sale_date"].max().date())

5. Sale date window                removed  410,251  ->    51,556 rows
date range kept: 2024-01-01 to 2026-09-23


## 6. Minimum sale price (removes $0, $1 and other nominal transfers)

In [7]:
b = len(df)
df = df[df["sale_price"] > RULES["min_sale_price"]]
record("6. Minimum sale price", f"sale_price > {RULES['min_sale_price']:,}", b, len(df))

6. Minimum sale price              removed   14,087  ->    37,469 rows


## 7. Flag issues the group chose to keep (no rows removed)
- `flag_condo`, `flag_vacant`: building code says condo or vacant land even though the category is SINGLE FAMILY.
- `bundle_size`: how many parcels in this cleaned set share the same sale date and price (1 = sold alone).
- `flag_zero_beds`, `flag_zero_baths`: 0 recorded (may mean missing).
- `flag_built_after_sale`: year built is later than the sale year.
- `price_to_assessed`: sale price / city market value (for checking odd sales; NOT a model input, it uses the target).

In [8]:
bc = df["building_code_description"].fillna("")
df["flag_condo"] = bc.str.contains("CONDO")
df["flag_vacant"] = bc.str.contains("VACANT")
df["bundle_size"] = df.groupby(["sale_date", "sale_price"])["parcel_number"].transform("count")
df["flag_zero_beds"] = df["number_of_bedrooms"].eq(0)
df["flag_zero_baths"] = df["number_of_bathrooms"].eq(0)
df["sale_year"] = df["sale_date"].dt.year
df["sale_month"] = df["sale_date"].dt.to_period("M").astype(str)
df["flag_built_after_sale"] = df["year_built"] > df["sale_year"]
df["price_to_assessed"] = (df["sale_price"] / df["market_value"]).round(3)

if RULES["drop_condo_and_vacant"]:
    b = len(df); df = df[~(df["flag_condo"] | df["flag_vacant"])]
    record("7a. Drop condo and vacant", "flag_condo or flag_vacant", b, len(df))
if RULES["drop_bundled_sales"]:
    b = len(df); df = df[df["bundle_size"] == 1]
    record("7b. Drop bundled sales", "bundle_size > 1", b, len(df))
record("7. Add flags (no rows removed)", "flags only", len(df), len(df))

flags = ["flag_condo", "flag_vacant", "flag_zero_beds", "flag_zero_baths", "flag_built_after_sale"]
print(df[flags].sum().to_string())
print("rows in bundles of 2+:", int((df["bundle_size"] > 1).sum()))

7. Add flags (no rows removed)     removed        0  ->    37,469 rows
flag_condo               2950
flag_vacant              1233
flag_zero_beds           2423
flag_zero_baths          2109
flag_built_after_sale      56
rows in bundles of 2+: 7897


## 8. Integrity checks (must all pass)

In [9]:
assert df["parcel_number"].is_unique, "duplicate parcels"
assert df["sale_price"].gt(RULES["min_sale_price"]).all()
assert df["sale_date"].ge(pd.Timestamp(RULES["min_sale_date"])).all()
assert (df["category_code_description"] == RULES["category"]).all()
assert not df.columns.isin(PERSONAL).any(), "personal columns still present"
print("all integrity checks passed:", df.shape)

all integrity checks passed: (37469, 80)


## 9. Remaining missing values in the cleaned set (decisions still open for the group)

In [10]:
key = ["sale_price", "total_livable_area", "number_of_bedrooms", "number_of_bathrooms", "year_built",
       "zip_code", "exterior_condition", "interior_condition", "quality_grade", "garage_spaces",
       "number_stories", "central_air", "basements", "number_of_rooms", "fireplaces"]
remaining = pd.DataFrame({"missing": df[key].isna().sum(),
                          "missing_pct": (df[key].isna().mean() * 100).round(1),
                          "zeros": (df[key] == 0).sum()})
remaining

,missing,missing_pct,zeros
sale_price,0,0.0,0
total_livable_area,4,0.0,2
number_of_bedrooms,92,0.2,2423
number_of_bathrooms,1275,3.4,2109
year_built,4,0.0,4
zip_code,1,0.0,0
exterior_condition,7,0.0,0
interior_condition,9,0.0,1
quality_grade,1929,5.1,0
garage_spaces,1154,3.1,25126


## 10. Save cleaned data and the cleaning log

In [11]:
front = ["parcel_number", "location", "zip_code", "census_tract", "sale_date", "sale_year", "sale_month",
         "sale_price", "market_value", "price_to_assessed", "category_code_description", "building_code_description",
         "total_livable_area", "total_area", "number_of_bedrooms", "number_of_bathrooms", "number_stories",
         "year_built", "year_built_estimate", "exterior_condition", "interior_condition", "quality_grade",
         "garage_spaces", "central_air", "basements", "fireplaces", "number_of_rooms",
         "flag_condo", "flag_vacant", "bundle_size", "flag_zero_beds", "flag_zero_baths", "flag_built_after_sale",
         "x_2272", "y_2272"]
clean = df[front + [c for c in df.columns if c not in front]].sort_values(["sale_date", "parcel_number"])
clean.to_csv(f"{OUT_DIR}/philly_sf_sales_clean.csv", index=False)
log_df = pd.DataFrame(log)
log_df.to_csv(f"{OUT_DIR}/cleaning_log.csv", index=False)
print("saved", clean.shape)
log_df

saved (37469, 80)


,step,rule,rows_before,rows_removed,rows_after
0,1. Load raw file,none,583780,0,583780
1,2. Organize (no rows removed),"drop personal columns, fix types",583780,0,583780
2,3. Category filter,category_code_description == SINGLE FAMILY,583780,120771,463009
3,4. Valid sale date,sale_date not missing or unparseable,463009,1202,461807
4,5. Sale date window,sale_date >= 2024-01-01,461807,410251,51556
5,6. Minimum sale price,"sale_price > 25,000",51556,14087,37469
6,7. Add flags (no rows removed),flags only,37469,0,37469
